# Stage 2 — complete experiment comparison

Scan the complete Stage 2 artifact tree, exclude incomplete LOSO runs, export a compact thesis-ready table, and show one matched representative comparison. The default representative setting is residual + fine-tune; change the two parameters below after reviewing final performance.


In [ ]:
from pathlib import Path
import os, subprocess, sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():  # VS Code may start in notebooks/
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Open this notebook from final_refactored or its notebooks folder.')
os.chdir(ROOT)
SRC_DIR = ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH'] = str(SRC_DIR) + os.pathsep + SUBPROCESS_ENV.get('PYTHONPATH', '')

ARTIFACTS = Path('artifacts/stage2')
OUTPUT = Path('results/tables/stage2/fusion_comparison')
EXPECTED_SUBJECTS = 154
REPRESENTATIVE_METHOD = 'residual'   # concatenation, film, or residual
REPRESENTATIVE_STRATEGY = 'finetune' # finetune or scratch

ARTIFACTS, REPRESENTATIVE_METHOD, REPRESENTATIVE_STRATEGY


## Build the complete comparison

Every completed configuration is identified by feature source, fusion method, training strategy, and target. The primary paired quantity is ΔRMSE = experiment RMSE − DRS-only RMSE for the same held-out subject; negative values favor the experiment. Runs with fewer than `EXPECTED_SUBJECTS` unique folds are reported and skipped.


In [ ]:
command = [
    sys.executable, '-m', 'subject_nirs.stage2.comparison',
    '--root', str(ARTIFACTS),
    '--output-dir', str(OUTPUT),
    '--expected-subjects', str(EXPECTED_SUBJECTS),
    '--representative-method', REPRESENTATIVE_METHOD,
    '--representative-strategy', REPRESENTATIVE_STRATEGY,
    '--no-pdf',
]
subprocess.run(command, cwd=ROOT, env=SUBPROCESS_ENV, check=True)


## Display the thesis-ready table and representative figure

`final_experiment_table.csv` retains numeric columns for later analysis. The formatted CSV is convenient for direct thesis-table review.


In [ ]:
numeric_table = pd.read_csv(OUTPUT / 'final_experiment_table.csv')
formatted_table = pd.read_csv(OUTPUT / 'final_experiment_table_formatted.csv')
display(formatted_table)
display(Image(filename=str(OUTPUT / 'representative_loso_comparison.png'), width=1300))

# Keep the numeric table available for sorting/filtering in later cells.
numeric_table
